# SkillForge — Train `board_pose.tflite` on Google Colab (T4 GPU)

This notebook trains a **YOLOv8n-pose** keypoint detection model on your Roboflow breadboard dataset and exports an **int8 quantized TFLite** model for on-device mobile inference in React Native / VisionCamera.

### Pipeline Steps:
1. **Check GPU Runtime** (Tesla T4)
2. **Install Ultralytics & Roboflow**
3. **Download Labeled Dataset** from Roboflow (Pre-configured!)
4. **Patch data.yaml** with kpt_shape if needed
5. **Train YOLOv8n-pose** (120 epochs)
6. **Validate & Inspect Results**
7. **Export to int8 TFLite** (`board_pose.tflite`)
8. **Download Model & Labelmap** for the mobile app

## 1. Verify GPU Allocation
Make sure your Colab runtime is set to **T4 GPU** (*Runtime* → *Change runtime type* → *T4 GPU*).

In [ ]:
!nvidia-smi

## 2. Install Ultralytics, Roboflow & Dependencies

In [ ]:
!pip install --upgrade ultralytics roboflow tensorflow opencv-python onnx pyyaml

## 3. Download Dataset from Roboflow (Pre-configured)

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="KrUxiAqCplSr35skpEoI")
project = rf.workspace("utkarsh-singh-ofhfi").project("skillforge-board-pose")
version = project.version(1)
dataset = version.download("yolov8")
print("Dataset downloaded successfully at:", dataset.location)

## 4. Patch data.yaml with Keypoint Shape
Newer Ultralytics versions require `kpt_shape` inside `data.yaml`, **not** as a train argument.
This cell ensures it's present.

In [ ]:
import os
import glob
import yaml

# Locate the data.yaml file
yaml_files = glob.glob(f"{dataset.location}/**/data.yaml", recursive=True)
if not yaml_files:
    yaml_files = glob.glob("**/data.yaml", recursive=True)
if not yaml_files:
    raise FileNotFoundError("Could not find data.yaml! Verify dataset download.")

data_yaml_path = os.path.abspath(yaml_files[0])
print(f"Found data.yaml at: {data_yaml_path}")

# Read current contents
with open(data_yaml_path, 'r') as f:
    data_cfg = yaml.safe_load(f)

# Patch kpt_shape if missing
if 'kpt_shape' not in data_cfg:
    data_cfg['kpt_shape'] = [6, 3]
    with open(data_yaml_path, 'w') as f:
        yaml.dump(data_cfg, f, default_flow_style=False)
    print('Patched data.yaml with kpt_shape: [6, 3]')
else:
    print(f'data.yaml already has kpt_shape: {data_cfg["kpt_shape"]}')

# Print full config for verification
print('\n--- data.yaml contents ---')
with open(data_yaml_path, 'r') as f:
    print(f.read())

## 5. Train YOLOv8n-pose for Breadboard Pose
Trains for 120 epochs with 6 keypoints. `kpt_shape` is read from `data.yaml` automatically.

In [ ]:
from ultralytics import YOLO

# Load lightweight YOLOv8n-pose pretrained weights
model = YOLO('yolov8n-pose.pt')

# Train on GPU — kpt_shape is in data.yaml, NOT passed here
results = model.train(
    data=data_yaml_path,
    epochs=120,
    imgsz=640,
    batch=16,
    device=0,
    project='skillforge',
    name='board_pose_v1',
    plots=True,
    save=True
)
print('Training Complete!')

## 6. Evaluate Validation Metrics

In [ ]:
from IPython.display import Image, display

# Display training loss and validation mAP curves
results_img = 'skillforge/board_pose_v1/results.png'
if os.path.exists(results_img):
    display(Image(filename=results_img))

# Display sample validation predictions
val_batch_img = 'skillforge/board_pose_v1/val_batch0_pred.jpg'
if os.path.exists(val_batch_img):
    display(Image(filename=val_batch_img))

## 7. Export to int8 Quantized TFLite
Converts the trained PyTorch weights to mobile-ready `.tflite` with int8 quantization for ultra-fast GPU/NNAPI inference on Android (iQOO 15) and iOS.

In [ ]:
best_weights = 'skillforge/board_pose_v1/weights/best.pt'
export_model = YOLO(best_weights)

# Export to TFLite (int8 quantized)
tflite_path = export_model.export(format='tflite', int8=True, imgsz=640)
print(f"TFLite Model Exported at: {tflite_path}")

## 8. Package and Download Model for Mobile App
Creates `board_pose.tflite` and `labelmap.json` and automatically triggers download to your computer.

In [ ]:
import json
import shutil
from google.colab import files

# 1. Create labelmap.json matching SkillForge frozen contract
labelmap = {
    "model_name": "board_pose",
    "version": "1.0.0",
    "classes": ["breadboard"],
    "keypoints": [
        {"id": 0, "name": "topLeft", "description": "Top-left outer corner"},
        {"id": 1, "name": "topRight", "description": "Top-right outer corner"},
        {"id": 2, "name": "bottomLeft", "description": "Bottom-left outer corner"},
        {"id": 3, "name": "bottomRight", "description": "Bottom-right outer corner"},
        {"id": 4, "name": "dividerLeft", "description": "Left end of center trench"},
        {"id": 5, "name": "dividerRight", "description": "Right end of center trench"}
    ]
}

with open('labelmap.json', 'w') as f:
    json.dump(labelmap, f, indent=2)

# 2. Locate generated tflite file
tflite_candidates = glob.glob('skillforge/board_pose_v1/weights/*best*_saved_model/*float16*.tflite') + \
                    glob.glob('skillforge/board_pose_v1/weights/*best*.tflite') + \
                    glob.glob('skillforge/board_pose_v1/weights/*best*_saved_model/*.tflite') + \
                    glob.glob('**/*.tflite', recursive=True)

found_tflite = None
for p in tflite_candidates:
    if 'best' in p and p.endswith('.tflite'):
        found_tflite = p
        break
if not found_tflite and tflite_candidates:
    found_tflite = tflite_candidates[0]

if found_tflite:
    shutil.copy(found_tflite, 'board_pose.tflite')
    print("Downloading board_pose.tflite and labelmap.json...")
    files.download('board_pose.tflite')
    files.download('labelmap.json')
else:
    print("Could not find .tflite directly, searching directory:")
    for root, dirs, fnames in os.walk('skillforge'):
        for fn in fnames:
            if fn.endswith('.tflite'):
                shutil.copy(os.path.join(root, fn), 'board_pose.tflite')
                files.download('board_pose.tflite')
                files.download('labelmap.json')
                break